# 도경 handoff → 젯슨 패키지  (2026-09-21 포맷 전용)

**셀 0 의 `BRANCH` 한 글자만 바꿔 사본에서 돌리면 두 갈래가 된다.**

```
A = 우리 파이프라인   dataset_trim0.json   episode_start_trim_s 0.0   GPU 2
B = 공식(현석 조건)   handoff dataset.json episode_start_trim_s 3.0   GPU 3
```

도경이 주는 형식은 고정이다. 셀 2 가 그 형식을 빌더·게이트가 읽는 형태로 맞춘다.
위에서 아래로 순서대로. **건너뛰는 셀 없다.**

In [ ]:
import os, sys, json, subprocess, time
from pathlib import Path

BRANCH = "A"      # ← "A" 우리(trim 0.0) · "B" 공식(trim 3.0). 사본에서 이 한 글자만 바꾼다

HOME    = Path.home()
REPO    = HOME / "S15P21A103"
BUILDER = HOME / "ubuntu_umi" / "02_umi_dataset_builder"
BUNDLE  = HOME / "hyeonseok" / "umi_gpu_bundle"
PY_GATE = HOME / "envs/handoff312/bin/python"     # 게이트·계측기 (numpy 2.x)
PY_BLD  = HOME / "envs/umi02/bin/python"          # 02 빌더 (numpy 1.26 고정)
PY_TRN  = "python"                                # 학습기 (시스템 파이썬, torch cu126)

# ── 도경 handoff 배치 ─────────────────────────────────────────
DATE     = "260921"
HANDOFF  = HOME / "data/onsite0921/orbslam_handoff_20260921"
RAW      = HOME / "data/onsite0921/raw_unzip/demonstrations_260921"
SEL      = HANDOFF / "episodes_selected"
MAPD     = HANDOFF / "mapping"
SLAM_TAG = MAPD / "tx_slam_tag.json"              # 셀 2 가 없으면 만든다
CAM_TCP  = BUILDER / "configs" / "s22_camera_tcp.json"

_B = {"A": ("ours",     REPO / "AI/configs/umi_builder/dataset_trim0.json", 0.0, "2"),
      "B": ("official", BUILDER / "configs" / "dataset.json",               3.0, "3")}
assert BRANCH in _B, f"BRANCH 는 A 또는 B. 받은 값 {BRANCH!r}"
_tag, DATASET_CFG, TRIM_EXPECT, GPU = _B[BRANCH]
NAME = f"onsite{DATE}_{BRANCH}_{_tag}"
OUT  = REPO / "out" / NAME
EPOCHS, BATCH = 120, 64
OUT.mkdir(parents=True, exist_ok=True)


def run(cmd, cwd=None, env=None, title=None):
    """Run a command, streaming output live. Returns the exit code.
    명령을 돌리며 출력을 실시간으로 흘린다. 반환은 종료코드."""
    if title:
        print(f"\u2500\u2500 {title}\n$ {' '.join(map(str, cmd))}\n", flush=True)
    e = dict(os.environ); e.update(env or {})
    p = subprocess.Popen([str(c) for c in cmd], cwd=cwd and str(cwd), env=e,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    print(f"\n[종료코드 {p.returncode}]", flush=True)
    return p.returncode


def jload(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))


print(f"갈래 {BRANCH} · {NAME}")
print(f"  trim 기대값 {TRIM_EXPECT} · GPU {GPU} · {EPOCHS} epoch · batch {BATCH}")
for k, v in [("REPO", REPO), ("BUILDER", BUILDER), ("BUNDLE", BUNDLE),
             ("RAW", RAW), ("SEL", SEL), ("MAPD", MAPD), ("SLAM_TAG", SLAM_TAG),
             ("CAM_TCP", CAM_TCP), ("DATASET_CFG", DATASET_CFG), ("OUT", OUT)]:
    print(f"  {k:12} {'있음' if Path(v).exists() else '없음'}  {v}")

## 1. 프리플라이트 — 없으면 여기서 멈춘다

In [ ]:
ok = tot = 0
def chk(name, cond, hint=""):
    global ok, tot
    tot += 1; ok += bool(cond)
    print(f"  [{'있음' if cond else '없음'}] {name}" + ("" if cond else f"   <- {hint}"))

print("필수 —")
chk("게이트용 python", PY_GATE.exists(), str(PY_GATE))
chk("빌더용 python",   PY_BLD.exists(),  "python3 -m venv ~/envs/umi02")
chk("02 빌더",         (BUILDER/"build_dataset.py").exists(), str(BUILDER))
chk("학습기",          (BUNDLE/"03_umi_policy_trainer/train_policy.py").exists(), str(BUNDLE))
for t in ("gate_slam_batch", "make_episode_plan", "audit_umi_zarr",
          "export_deploy_ckpt", "smoke_deploy_ckpt", "make_jetson_package"):
    chk(f"계측기 {t}", (REPO/"AI/tools"/f"{t}.py").exists(), "git pull origin ai")
chk("DATASET_CFG", DATASET_CFG.exists(), str(DATASET_CFG))
chk("handoff raw",  RAW.exists(),  str(RAW))
chk("handoff 편",   SEL.exists(),  str(SEL))
chk("handoff 매핑", MAPD.exists(), str(MAPD))
print(f"\n필수 {ok} / {tot}")
assert ok == tot, "위 [없음] 을 먼저 해결한다. 미판정은 통과가 아니다"

# config 가 기대한 trim 값인지 되읽는다 — 파일명만 보고 믿지 않는다
_cfg = jload(DATASET_CFG)
print(f"\nDATASET_CFG episode_start_trim_s = {_cfg.get('episode_start_trim_s')}")
assert abs(float(_cfg["episode_start_trim_s"]) - TRIM_EXPECT) < 1e-9, \
    f"갈래 {BRANCH} 기대 {TRIM_EXPECT} 인데 config 는 {_cfg.get('episode_start_trim_s')} 다"
print("갈래와 config 가 맞다")

## 2. handoff 적응 — 도경 형식을 빌더·게이트가 읽는 형태로

도경이 주는 것과 우리가 필요한 것의 차이 4가지를 여기서 메운다. **전부 멱등**이다.

| 도경이 주는 것 | 필요한 것 | 처리 |
|---|---|---|
| 편 직하 `camera_trajectory.csv` | `<편>/slam/camera_trajectory.csv` | 링크 |
| `trajectory_validation_v10.json` | `slam/trajectory_validation.json` | 이름 맞춰 링크 |
| 없음 | `gripper_width.csv` · `telemetry.json` | raw 에서 `prepare` 생성 |
| 없음 | `tx_slam_tag.json` | 매핑 편으로 `slam-tag` 생성 |

`prepare` 는 편당 15~25초다. 96편이면 처음 한 번에 30분 안쪽. 이미 된 편은 건너뛴다.

In [ ]:
LINKS = [("trajectory_validation_v10.json", "trajectory_validation.json")] + \
        [(x, x) for x in ("camera_trajectory.csv", "camera_trajectory_tum.txt",
                          "gripper_width.csv", "telemetry.json",
                          "gripper_report.json", "prepare_report.json")]

def relink(d: Path) -> int:
    """Mirror handoff files into <ep>/slam/ under the names the gate reads.
    handoff 파일을 게이트가 읽는 이름으로 slam/ 에 건다."""
    (d / "slam").mkdir(exist_ok=True)
    n = 0
    for a, b in LINKS:
        s, t = d / a, d / "slam" / b
        if s.exists() and not t.exists():
            t.symlink_to(s.resolve()); n += 1
    return n

# ── 매핑 편 특정 — 추측하지 않고 리포트에서 읽는다 ──
oc = jload(HANDOFF / "reports/collection_outcomes.json")
rows = oc["episodes"] if isinstance(oc, dict) else oc
maps = [r["episode_id"] for r in rows if r.get("role") == "mapping"]
print(f"role==mapping {len(maps)} / 리포트 행 {len(rows)}")
assert len(maps) == 1, f"매핑 편이 1개가 아니다: {maps}"
MAP_REC = RAW / maps[0]
print(f"매핑 편 {MAP_REC.name}  {'있음' if MAP_REC.exists() else '없음'}")
assert MAP_REC.exists(), "raw 에 매핑 편이 없다"

# ── tx_slam_tag ──
print(f"\nmapping/slam 링크 {relink(MAPD)}개")
if SLAM_TAG.exists():
    print(f"tx_slam_tag 이미 있음 — status {jload(SLAM_TAG).get('status')}")
else:
    rc = run([PY_BLD, "build_dataset.py", "slam-tag", MAP_REC, MAPD],
             cwd=BUILDER, title="tx_slam_tag 생성")
    assert rc == 0 and SLAM_TAG.exists(), "slam-tag 실패"
_t = jload(SLAM_TAG)
print(f"  status {_t.get('status')} · tag {_t.get('tag_id')} / {_t.get('tag_size_m')}m "
      f"· 태그 보인 프레임 {_t.get('frames_with_tag')} / 추적 {_t.get('tracked_frames')}")
assert _t.get("status") == "pass", "tx_slam_tag 가 pass 가 아니다"

In [ ]:
# ── 편별 prepare + 링크 ──
eps = sorted(p for p in SEL.iterdir() if p.is_dir())
todo = [d for d in eps if not (d / "gripper_width.csv").exists()]
print(f"편 {len(eps)} · prepare 필요 {len(todo)} · 이미 된 편 {len(eps) - len(todo)}")

t0 = time.time()
fails = []
for i, d in enumerate(todo, 1):
    r = subprocess.run([str(PY_BLD), "build_dataset.py", "prepare", str(RAW / d.name), str(d)],
                       cwd=str(BUILDER), capture_output=True, text=True)
    if r.returncode != 0:
        fails.append((d.name, r.stderr.strip().splitlines()[-1] if r.stderr.strip() else "?"))
    if i % 10 == 0 or i == len(todo):
        print(f"  {i}/{len(todo)} · 경과 {time.time()-t0:.0f}s · 실패 {len(fails)}", flush=True)
for n, why in fails[:5]:
    print(f"  실패 {n}: {why}")

n_link = sum(relink(d) for d in eps)
ready = sum((d / "slam/gripper_width.csv").exists() and
            (d / "slam/trajectory_validation.json").exists() for d in eps)
print(f"\n링크 신규 {n_link} · 준비된 편 {ready} / 전체 {len(eps)} · prepare 실패 {len(fails)}")
assert ready == len(eps), "준비 안 된 편이 있다 — 위 실패 사유를 본다. 미판정은 통과가 아니다"

## 3. SLAM 게이트

⚠️ **2026-09-21 — 마커 검출률·결손 구간·키프레임 수를 필수에서 권고로 내렸다.**

| | |
|---|---|
| 왜 | 오늘 수집은 마커 **간헐 노출** 프로토콜이다(사용자 진술). 0.80 의 근거였던 현석 v4 0.847 은 마커 상시 노출 조건이라 전제가 다르다 |
| 실측 | 전체 검출률 최대 0.89 · 중앙 0.74 (n=45). 파지 순간 창(±15프레임) 검출률 중앙 0.71 — **창 기준으로 바꿔도 수율이 거의 안 는다** |
| 키프레임 | `trajectory_validation_v10` 스키마에 키 자체가 없다 → 영구 미판정 |
| 미검증 | **gap 채널 보간이 학습에 미치는 영향은 안 쟀다.** LIMITS 기록 대상 |

게이트를 낮춘 게 아니라 전제가 깨진 기준을 다시 정한 것이다. 근거가 바뀌면 되돌린다.

In [ ]:
rc = run([PY_GATE, REPO/"AI/tools/gate_slam_batch.py",
          "--processed", SEL, "--out", OUT/"slam_gate.json"], title="SLAM 게이트")
assert rc == 0
g = jload(OUT/"slam_gate.json"); t = g["tally"]
print(f"\n>> 통과 {t.get('PASS',0)} · 불합격 {t.get('FAIL',0)} · "
      f"미판정 {t.get('INCOMPLETE',0)} / 전체 {g['total']}")
for r in g["batch"]["rows"]:
    print("   배치:", r["name"], "->", r["got"])
assert t.get("PASS", 0) > 0, "통과 0편. 데이터가 아니라 기준을 먼저 의심한다"

## 4. episode plan — 게이트 통과 편만

In [ ]:
rc = run([PY_GATE, REPO/"AI/tools/make_episode_plan.py",
          "--gate", OUT/"slam_gate.json", "--raw", RAW, "--processed", SEL,
          "--slam-tag", SLAM_TAG, "--out", OUT/"plan.json"], title="plan 생성")
assert rc == 0, "plan 실패 — 위 게이트 결과와 경로를 본다"
pl = jload(OUT/"plan.json")
print(f"\n>> plan 편 {len(pl['episodes'])} / 게이트 통과 {g['tally'].get('PASS',0)}")
assert len(pl["episodes"]) > 0

## 5. zarr 빌드

빌더는 덮어쓰기를 거부한다. 다시 돌리려면 기존 zarr 을 먼저 지운다.
`--dataset-config` 가 먹었는지 **report 를 되읽어** 확인한다 — 안 먹으면 기본값 3.0 이 조용히 들어온다.

In [ ]:
ZARR   = OUT / f"ds_{NAME}.zarr.zip"
REPORT = OUT / f"ds_{NAME}.zarr.report.json"
assert not ZARR.exists(), f"이미 있다. 지우고 다시 돌려라: {ZARR}"

rc = run([PY_BLD, "build_dataset.py", "build", OUT/"plan.json", ZARR,
          "--dataset-config", DATASET_CFG, "--camera-tcp", CAM_TCP],
         cwd=BUILDER, title="zarr 빌드 (영상 디코딩 — 수십 분)")
assert rc == 0, "빌드 실패. 로그 위쪽의 편 이름과 사유를 본다"
# 종료코드 0 만 보면 '성공했는데 파일이 없다' 가 통과한다
assert ZARR.exists(),   f"종료코드 0 인데 zarr 가 없다: {ZARR}"
assert REPORT.exists(), f"종료코드 0 인데 report 가 없다: {REPORT}"

print(f"\nzarr {ZARR}  {ZARR.stat().st_size:,} B")
d = jload(REPORT)
for k in ("status","episodes","frames","native_sample_rate_hz","world_frame",
          "episode_start_trim_s","camera_tcp_status","training_input_status",
          "physical_deployment_ready"):
    print(f"  {k:28} {d.get(k)}")
assert abs(float(d.get("episode_start_trim_s")) - TRIM_EXPECT) < 1e-9, \
    f"trim 이 {d.get('episode_start_trim_s')} 다 — 갈래 {BRANCH} 기대는 {TRIM_EXPECT}"
print(f"\ntrim {TRIM_EXPECT} 확인 — 갈래 {BRANCH} 데이터가 맞다")

## 6. 데이터 감사

In [ ]:
rc = run([PY_GATE, REPO/"AI/tools/audit_umi_zarr.py", "--zarr", ZARR,
          "--out", OUT/"audit.json"], title="zarr 감사")
print("감사 종료코드", rc, "— 불합격이면 아래 학습으로 넘어가지 않는다")
assert rc == 0

## 7. 학습

120 epoch 이 약 2.6시간이다. `nohup` 이 아니라 셀에서 직접 돌린다 —
브라우저를 닫아도 커널은 산다. **커널 Shutdown/Restart 만 하지 마라.**

In [ ]:
RUN_ID  = f"{NAME}_{time.strftime('%H%M%S')}"
TRAINER = BUNDLE / "03_umi_policy_trainer" / "train_policy.py"
PROFILE = BUNDLE / "03_umi_policy_trainer" / "configs" / "policy_resnet18_gpu.yaml"
env = {"CUDA_VISIBLE_DEVICES": GPU, "UMI_ROOT": str(BUNDLE/"third_party/umi"),
       "WANDB_MODE": "disabled", "HYDRA_FULL_ERROR": "1"}
print(f"run-id {RUN_ID} · GPU {GPU}")

rc = run([PY_TRN, TRAINER, "check", ZARR], cwd=BUNDLE, env=env, title="dataset check")
assert rc == 0, "dataset check 실패 — 학습으로 넘어가지 않는다"

rc = run([PY_TRN, TRAINER, "train", ZARR, "--run-id", RUN_ID, "--runs-dir", OUT/"runs",
          "--epochs", EPOCHS, "--batch", BATCH, "--eval-batch", BATCH, "--profile", PROFILE],
         cwd=BUNDLE, env=env, title=f"학습 {EPOCHS} epoch")
assert rc == 0

## 8. 배포용 변환 + 검사

In [ ]:
RUN_DIR = OUT / "runs" / RUN_ID
print("run 디렉터리", RUN_DIR, "있음" if RUN_DIR.exists() else "없음")
assert RUN_DIR.exists(), "run 디렉터리가 없다 — 학습 로그를 본다"

rc = run([PY_GATE, REPO/"AI/tools/export_deploy_ckpt.py", "--run-dir", RUN_DIR,
          "--out-dir", OUT/"deploy"], title="배포용 변환")
assert rc == 0

rc = run([PY_GATE, REPO/"AI/tools/smoke_deploy_ckpt.py", "--deploy-dir", OUT/"deploy",
          "--out", OUT/"smoke.json"], title="배포 스모크")
assert rc == 0, "스모크 불합격 — 젯슨으로 안 보낸다"

## 9. 젯슨 패키지

`encoder.pt` · `denoiser.pt` · `metadata.json` · `dataset.report.json` 를 만들고 검사한다.
`robot_execution_enabled` 는 **뜻을 모르니 판정하지 않고 값만 찍는다.**

In [ ]:
rc = run([PY_GATE, REPO/"AI/tools/make_jetson_package.py", "--run-dir", RUN_DIR,
          "--out-dir", OUT/"jetson", "--dataset-report", REPORT],
         title="젯슨 패키지")
assert rc == 0, "젯슨 패키지 불합격 — 위 행을 본다"
for p in sorted((OUT/"jetson").rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(OUT/'jetson')}  {p.stat().st_size:,} B")
print(f"\n갈래 {BRANCH} 끝. 젯슨으로 보낼 것: {OUT/'jetson'}")

## 한계 — 인용할 때 반드시 붙인다

- **실물 로봇 구동 0회.** 모터 고장으로 전부 시뮬·데이터 검사다
- **gap 채널 보간이 학습에 미치는 영향 미측정.** 마커 검출률 중앙 0.74 에서 그대로 학습했다
- **camera_tcp 는 물리검증판이 아니다.** 현석의 `4e16c0a2…` 가 오면 다시 빌드해 비교해야 한다
- 학습 실행 간 성공률 편차 **4.1% (n=2)** — 4% 미만 차이는 시드 노이즈와 구분 못 한다
- A/B 차이는 `episode_start_trim_s` **하나**다. 다른 인자를 바꾸면 비교가 깨진다